# 03 — Exploratory Data Analysis (EDA)

## Objective

Explore the Silver table to build
intuition about the 115 features before moving into the Gold layer and model
training.

The 115 features have no published semantic names in the public Kitsune dataset.
According to the dataset documentation, they represent 23 underlying traffic
statistics, each computed over 5 different time windows (23 × 5 = 115), but the
exact mapping from column index to statistic/window is not available. 

## Known context going in

- **Severe class imbalance**: 2,764,238 normal rows (99.75%) vs. 7,038 attack
  rows (0.25%), confirmed in `02_silver_transformation`.
- **No nulls, no duplicate row_ids**: data integrity already validated in Silver.

## Scope of this notebook

1. **General profile**: aggregate statistics (mean, stddev, min, max) across all
   115 features, computed on the full dataset — not a sample, since Spark can
   aggregate at scale without loading everything into memory.
2. **Feature ranking by discriminative power**: for each feature, measure how
   different its distribution looks between `label=0` and `label=1`, using a
   normalized mean difference (similar in spirit to a z-score / Cohen's d).
   This identifies which features actually separate normal traffic from attack
   traffic, without needing to know what each feature physically measures.
3. **Detailed visualization**: plot distributions for the top-ranked features
   only (not all 115 — with this many columns, visualizing everything adds
   noise, not insight).



In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("kitsune_project.silver_layer.syn_dos_clean")

print(f"Row count: {silver_df.count()}")
print(f"Column count: {len(silver_df.columns)}")

In [0]:
feature_columns = [c for c in silver_df.columns if c.startswith("feature_")]

# Calculation of the mean and standard deviation for each feature
stats_df = silver_df.groupBy("label").agg(
    *[F.mean(c).alias(f"mean_{c}") for c in feature_columns],
    *[F.stddev(c).alias(f"stddev_{c}") for c in feature_columns]
)

stats_pd = stats_df.toPandas()

In [0]:
import pandas as pd

# Separate the two rows: one for label=0, one for label=1
row_normal = stats_pd[stats_pd["label"] == 0].iloc[0]
row_attack = stats_pd[stats_pd["label"] == 1].iloc[0]

# Build the score for each feature
scores = []
for c in feature_columns:
    mean_normal = row_normal[f"mean_{c}"]
    mean_attack = row_attack[f"mean_{c}"]
    stddev_normal = row_normal[f"stddev_{c}"]

    score = abs(mean_attack - mean_normal) / stddev_normal
    scores.append({"feature": c, "score": score})

scores_df = pd.DataFrame(scores).sort_values("score", ascending=False)
scores_df.head(10)

## Feature ranking: which columns actually distinguish attack from normal traffic

For each of the 115 features, calculated a discriminative score:
`score = [mean(attack) - mean(normal)] / stddev(normal)`

A high score means that feature looks very different between normal traffic
and SYN DoS attacks. A score near 0 means the feature barely changes.

**Top results:**

| Feature | Score |
|---|---|
| feature_79 | 27.2 |
| feature_76 | 25.1 |
| feature_77 | 23.0 |
| feature_80 | 22.6 |
| feature_73 | 16.7 |
| feature_67 | 15.8 |
| feature_70 | 15.8 |
| feature_74 | 4.0 |

There is a clear drop after `feature_74` (score 4.0) down to the rest of the
features.

In [0]:
# Array of top features
top_features = ["feature_79", "feature_76", "feature_77", "feature_80", "feature_73", "feature_67", "feature_70", "feature_74"]

# Separate the two rows: one for label=0, one for label=1
normal_df = silver_df.filter(silver_df.label == 0)
attack_df = silver_df.filter(silver_df.label == 1)

normal_count = normal_df.count()

# Take a small sample of each
normal_sample = normal_df.sample(fraction=10000 / normal_count).toPandas() # bring only 10k normal
attack_sample = attack_df.toPandas()  # bring all the attack data since it's small

In [0]:
import numpy as np

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, feature in enumerate(top_features):
    # Log scale can't handle 0 or negative values — add 1 before taking the log
    normal_vals = np.log1p(normal_sample[feature])
    attack_vals = np.log1p(attack_sample[feature])

    axes[i].hist(normal_vals, bins=50, alpha=0.5, label="Normal", color="steelblue", density=True)
    axes[i].hist(attack_vals, bins=50, alpha=0.5, label="Attack", color="firebrick", density=True)
    axes[i].set_title(feature)
    axes[i].set_xlabel("log(value + 1)")
    axes[i].legend()

plt.tight_layout()
plt.show()

**Why log scale:** We use log scale because these features range from
values close to 0 up to extremely large numbers. With
a linear scale, all the small values got compressed into a single bar near
zero and we couldn't see any shape in the distribution. The log transform
fixes this by making each order of magnitude take up the same visual space.

**What the plots show:**
For most features, the normal and attack histograms overlap heavily near
zero — both classes have many low values there. But there's a tail of high
values (around log=20 to log=40) where only attack traffic appears; normal
traffic never reaches those values. In feature_80, the pattern is different:
attack traffic concentrates strongly around log≈11-13, in a range where
normal traffic is much more spread out.

**Implication for modeling:**
It will be difficult for a simple model to separate the two classes cleanly,
because most attack packets overlap with normal ones near zero. There's no
single threshold that cleanly divides "normal" from "attack" — only a
minority of attack packets stand out with extreme values.